# Transport Remodeling Exploration


## 1. Setup

Run this notebook from either the repository root or the `transport_remodeling` directory.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)
pd.set_option('display.width', 200)


def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / 'transport_remodeling.py').exists() and (candidate / 'demo_data').exists():
            return candidate
    raise RuntimeError('Could not find project root containing transport_remodeling.py and demo_data/.')


PROJECT_ROOT = find_project_root()
TRANSPORT_DIR = PROJECT_ROOT
if str(TRANSPORT_DIR) not in sys.path:
    sys.path.insert(0, str(TRANSPORT_DIR))

import transport_remodeling as tr
tr = importlib.reload(tr)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRANSPORT_DIR:', TRANSPORT_DIR)


## 2. Run Configuration

Choose one configured healthy/disease comparison. Clustering methods are assigned per input table because healthy and disease outputs may come from different directories or clustering choices.

In [ ]:
RUNS = {
    'condition_shift_example': {
        'organ': 'synthetic_tissue',
        'healthy_cluster_csv': PROJECT_ROOT / 'demo_data/healthy_cluster_library.csv',
        'healthy_clustering_method': 'leiden',
        'disease_cluster_csv': PROJECT_ROOT / 'demo_data/disease_cluster_library.csv',
        'disease_clustering_method': 'leiden',
    },
}

RUN_NAME = 'condition_shift_example'
CONFIG = RUNS[RUN_NAME]
OUT_DIR = PROJECT_ROOT / 'outputs' / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_summary = {
    'run_name': RUN_NAME,
    'organ': CONFIG['organ'],
    'healthy_cluster_csv': str(Path(CONFIG['healthy_cluster_csv']).resolve()),
    'healthy_clustering_method': CONFIG['healthy_clustering_method'],
    'disease_cluster_csv': str(Path(CONFIG['disease_cluster_csv']).resolve()),
    'disease_clustering_method': CONFIG['disease_clustering_method'],
    'out_dir': str(OUT_DIR),
}
run_summary


## 3. Cell-Type Harmonization

Edit the organ-specific canonical map before running the model. Raw `Comp_*` labels mapped to the same canonical label are summed.

In [ ]:
canonical_cell_map_synthetic_tissue = {}

CANONICAL_MAPS = {
    'synthetic_tissue': canonical_cell_map_synthetic_tissue,
}

canonical_cell_map = CANONICAL_MAPS.get(CONFIG['organ'], {})
print('organ:', CONFIG['organ'])
print('n canonical mappings:', len(canonical_cell_map))


## 4. Load Cluster Tables

The same cell-type map is applied to healthy and disease tables so the cost matrix uses a shared composition vocabulary.

In [ ]:
healthy_df = tr.load_cluster_metrics(CONFIG['healthy_cluster_csv'], canonical_map=canonical_cell_map)
disease_df = tr.load_cluster_metrics(CONFIG['disease_cluster_csv'], canonical_map=canonical_cell_map)

preview_cols = ['cluster_id', 'N_Cells', 'N_Slides', 'Niche_Signature', 'Top_Enriched_Cells (log2FC)']
healthy_preview_cols = [c for c in preview_cols if c in healthy_df.columns]
disease_preview_cols = [c for c in preview_cols if c in disease_df.columns]

print('healthy clusters:', healthy_df.shape[0])
print('disease clusters:', disease_df.shape[0])
print('healthy Comp_* columns:', sum(c.startswith('Comp_') for c in healthy_df.columns))
print('disease Comp_* columns:', sum(c.startswith('Comp_') for c in disease_df.columns))

display(healthy_df[healthy_preview_cols].head())
display(disease_df[disease_preview_cols].head())

### 4.1 Cell-Type Counts Before And After Harmonization

Count how many positive `Comp_*` cell-type composition columns are present before and after applying the notebook-defined harmonization map.


In [ ]:
def positive_comp_celltypes(df, tol=1e-12):
    celltypes = set()
    for col in [c for c in df.columns if str(c).startswith('Comp_')]:
        total = pd.to_numeric(df[col], errors='coerce').fillna(0.0).sum()
        if total > tol:
            celltypes.add(str(col).replace('Comp_', '', 1))
    return celltypes


raw_healthy_df = pd.read_csv(CONFIG['healthy_cluster_csv'])
raw_disease_df = pd.read_csv(CONFIG['disease_cluster_csv'])

healthy_before = positive_comp_celltypes(raw_healthy_df)
disease_before = positive_comp_celltypes(raw_disease_df)
healthy_after = positive_comp_celltypes(healthy_df)
disease_after = positive_comp_celltypes(disease_df)

celltype_count_summary = pd.DataFrame([
    {
        'metric': 'healthy_celltypes',
        'before_harmonization': len(healthy_before),
        'after_harmonization': len(healthy_after),
    },
    {
        'metric': 'disease_celltypes',
        'before_harmonization': len(disease_before),
        'after_harmonization': len(disease_after),
    },
    {
        'metric': 'shared_celltypes',
        'before_harmonization': len(healthy_before & disease_before),
        'after_harmonization': len(healthy_after & disease_after),
    },
])
celltype_count_summary['change_after_harmonization'] = (
    celltype_count_summary['after_harmonization']
    - celltype_count_summary['before_harmonization']
)

display(celltype_count_summary)


## 5. Model Options

The primary transport cost is composition-only. `n_cells` mass is the default. h5ad inputs are only needed for optional slide-mean mass.

In [ ]:
options = tr.TransportOptions(
    mass_mode='n_cells',
    healthy_h5ad=None,
    disease_h5ad=None,
    healthy_cluster_key=CONFIG['healthy_clustering_method'],
    disease_cluster_key=CONFIG['disease_clustering_method'],
    slide_key='slide',
    lambda_expand=0.35,
    lambda_reduce=0.35,
    lambda_residual=0.80,
    branch_min_total_mass=0.01,
    branch_min_source_fraction=0.10,
    unknown_fraction_threshold=0.30,
    pseudocount_mass=1e-6,
    plan_mass_tol=1e-10,
    mass_zero_tol=1e-9,
    top_n_heatmap=25,
)

asdict(options)

## 6. Build Features, Masses, And Costs

This constructs the aligned composition matrices, abundance mass vectors, and scaled composition cost matrix used by OT.

In [ ]:
healthy_comp, disease_comp, comp_cols = tr.aligned_composition_matrices(healthy_df, disease_df)
healthy_mass, disease_mass = tr.compute_masses(healthy_df, disease_df, options)
total_cost, cost_long, cost_metadata = tr.build_cost_matrices(
    healthy_comp,
    disease_comp,
    healthy_df,
    disease_df,
    options,
)

print('healthy mass sum:', healthy_mass.sum())
print('disease mass sum:', disease_mass.sum())
print('cost shape:', total_cost.shape)
print('cost metadata:', cost_metadata)

display(pd.DataFrame({
    'condition': ['healthy', 'disease'],
    'n_clusters': [len(healthy_mass), len(disease_mass)],
    'min_mass': [healthy_mass.min(), disease_mass.min()],
    'max_mass': [healthy_mass.max(), disease_mass.max()],
}))

display(cost_long.sort_values('total_cost').head(10))

## 7. Default OT Run

The default run records the behavior of the configured penalties. If expansion, reduction, and residual mass are near zero, the default setting is behaving like balanced transport.

In [ ]:
solution = tr.solve_unbalanced_transport(
    total_cost,
    healthy_mass,
    disease_mass,
    options.lambda_expand,
    options.lambda_reduce,
    options.lambda_residual,
)
default_components = tr.objective_components(
    solution,
    total_cost,
    options.lambda_expand,
    options.lambda_reduce,
    options.lambda_residual,
)

print('solver status:', solution.solver_status)
print('solver message:', solution.solver_message)
display(pd.DataFrame([default_components]))

## 8. Summarize Transport Events

These are the main interpretation tables: source-level remodeling events, disease-target explanation, and nonzero transport links.

In [ ]:
source_summary = tr.summarize_sources(
    solution,
    total_cost,
    healthy_df,
    disease_df,
    healthy_mass,
    options,
)

target_summary = tr.summarize_targets(
    solution,
    total_cost,
    healthy_df,
    disease_df,
    disease_mass,
    options,
)

plan_df = tr.build_plan_df(
    solution,
    cost_long,
    healthy_df,
    disease_df,
    healthy_mass,
    disease_mass,
    total_cost,
    options,
)

print('nonzero transport links:', len(plan_df))
display(source_summary.head(10))
display(target_summary.head(10))

In [ ]:
plan_df

## 9. Write Core Outputs

The notebook writes reusable tables and run metadata. The final dashboard is displayed inline only and is not saved as a separate report.

In [ ]:
plan_df.to_csv(OUT_DIR / 'transport_plan.csv', index=False)
source_summary.to_csv(OUT_DIR / 'healthy_source_transport_summary.csv', index=False)
target_summary.to_csv(OUT_DIR / 'disease_target_transport_summary.csv', index=False)
source_summary.to_csv(OUT_DIR / 'transport_event_ranking.csv', index=False)
cost_long.to_csv(OUT_DIR / 'cost_matrix.csv', index=False)

tr.write_json(
    OUT_DIR / 'transport_parameters.json',
    {
        'run_name': RUN_NAME,
        'organ': CONFIG['organ'],
        'healthy_clustering_method': CONFIG['healthy_clustering_method'],
        'disease_clustering_method': CONFIG['disease_clustering_method'],
        'healthy_cluster_csv': str(CONFIG['healthy_cluster_csv']),
        'disease_cluster_csv': str(CONFIG['disease_cluster_csv']),
        'out_dir': str(OUT_DIR),
        'options': asdict(options),
        'cost_metadata': cost_metadata,
        'solver': {
            'objective': solution.objective,
            'status': solution.solver_status,
            'message': solution.solver_message,
        },
        'objective_components': default_components,
        'n_healthy_sources': int(len(healthy_df)),
        'n_disease_targets': int(len(disease_df)),
        'n_composition_features': int(len(comp_cols)),
        'composition_features': [c.replace('Comp_', '', 1) for c in comp_cols],
        'harmonization': {
            'defined_in_notebook': True,
            'canonical_map_size': int(len(canonical_cell_map)),
            'canonical_map': canonical_cell_map,
        },
    },
)

print('Wrote core outputs to:', OUT_DIR)

## 10. Penalty Path

Vary `lambda_expand = lambda_reduce` while holding `lambda_residual` fixed. This shows when the model permits source deviation versus collapses toward balanced OT.

In [ ]:
SHIFT_VALUES = [0.0, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.075, 0.10, 0.15, 0.20, 0.35]
PATH_RESIDUAL = options.lambda_residual
path_rows = []
source_path_rows = []
target_path_rows = []

for lam in SHIFT_VALUES:
    path_options = tr.TransportOptions(
        mass_mode=options.mass_mode,
        healthy_h5ad=options.healthy_h5ad,
        disease_h5ad=options.disease_h5ad,
        healthy_cluster_key=CONFIG['healthy_clustering_method'],
        disease_cluster_key=CONFIG['disease_clustering_method'],
        slide_key=options.slide_key,
        lambda_expand=lam,
        lambda_reduce=lam,
        lambda_residual=PATH_RESIDUAL,
        branch_min_total_mass=options.branch_min_total_mass,
        branch_min_source_fraction=options.branch_min_source_fraction,
        unknown_fraction_threshold=options.unknown_fraction_threshold,
        pseudocount_mass=options.pseudocount_mass,
        plan_mass_tol=options.plan_mass_tol,
        mass_zero_tol=options.mass_zero_tol,
        top_n_heatmap=options.top_n_heatmap,
    )
    path_solution = tr.solve_unbalanced_transport(total_cost, healthy_mass, disease_mass, lam, lam, PATH_RESIDUAL)
    path_source = tr.summarize_sources(path_solution, total_cost, healthy_df, disease_df, healthy_mass, path_options)
    path_target = tr.summarize_targets(path_solution, total_cost, healthy_df, disease_df, disease_mass, path_options)
    parts = tr.objective_components(path_solution, total_cost, lam, lam, PATH_RESIDUAL)
    top_net = path_source.sort_values(['net_shift_mass', 'healthy_cluster_id'], ascending=[False, True]).iloc[0]
    top_cost = path_source.sort_values(['cost_weighted_mass', 'healthy_cluster_id'], ascending=[False, True]).iloc[0]
    top_frag = path_source.sort_values(['split_entropy', 'healthy_cluster_id'], ascending=[False, True]).iloc[0]
    path_rows.append({
        'lambda_shift': lam,
        **parts,
        'top_net_shift_source': str(top_net['healthy_cluster_id']),
        'top_net_shift_mass': float(top_net['net_shift_mass']),
        'top_cost_weighted_source': str(top_cost['healthy_cluster_id']),
        'top_cost_weighted_mass': float(top_cost['cost_weighted_mass']),
        'top_fragmented_source': str(top_frag['healthy_cluster_id']),
        'top_split_entropy': float(top_frag['split_entropy']),
    })
    keep_source_cols = [
        'healthy_cluster_id', 'healthy_mass', 'transported_out_mass', 'net_shift_mass',
        'log2_transport_shift', 'mean_transport_cost', 'cost_weighted_mass',
        'n_disease_branches', 'split_entropy', 'effective_n_branches', 'event_labels',
    ]
    source_tmp = path_source[keep_source_cols].copy()
    source_tmp.insert(0, 'lambda_shift', lam)
    source_path_rows.append(source_tmp)

    keep_target_cols = [
        'disease_cluster_id', 'disease_mass', 'explained_transport_mass',
        'residual_mass', 'residual_fraction', 'dominant_healthy_source',
        'dominant_source_fraction_of_target', 'mean_incoming_cost',
        'mixed_source_entropy', 'effective_n_sources', 'unknown_fraction',
        'high_unknown_flag', 'residual_rank', 'incoming_cost_rank',
        'mixed_source_rank', 'disease_signature',
    ]
    target_tmp = path_target[keep_target_cols].copy()
    target_tmp.insert(0, 'lambda_shift', lam)
    target_path_rows.append(target_tmp)

lambda_path_df = pd.DataFrame(path_rows)
source_path_df = pd.concat(source_path_rows, ignore_index=True)
target_path_df = pd.concat(target_path_rows, ignore_index=True)

lambda_path_df.to_csv(OUT_DIR / 'penalty_path_summary.csv', index=False)
source_path_df.to_csv(OUT_DIR / 'penalty_path_source_events.csv', index=False)
target_path_df.to_csv(OUT_DIR / 'penalty_path_target_events.csv', index=False)

display(lambda_path_df)


## 11. Moderate Source-Regularized Diagnostic Run

This diagnostic run uses a moderate source-deviation penalty. It is useful for selecting follow-up events when the default setting is too close to balanced transport.

In [ ]:
CHOSEN_LAMBDA_SHIFT = 0.05
lambda_tag = str(CHOSEN_LAMBDA_SHIFT).replace('.', 'p')
CHOSEN_OUT_DIR = OUT_DIR / f'source_regularized_lambda_{lambda_tag}'
CHOSEN_OUT_DIR.mkdir(parents=True, exist_ok=True)

chosen_options = tr.TransportOptions(
    mass_mode=options.mass_mode,
    healthy_h5ad=options.healthy_h5ad,
    disease_h5ad=options.disease_h5ad,
    healthy_cluster_key=CONFIG['healthy_clustering_method'],
    disease_cluster_key=CONFIG['disease_clustering_method'],
    slide_key=options.slide_key,
    lambda_expand=CHOSEN_LAMBDA_SHIFT,
    lambda_reduce=CHOSEN_LAMBDA_SHIFT,
    lambda_residual=PATH_RESIDUAL,
    branch_min_total_mass=options.branch_min_total_mass,
    branch_min_source_fraction=options.branch_min_source_fraction,
    unknown_fraction_threshold=options.unknown_fraction_threshold,
    pseudocount_mass=options.pseudocount_mass,
    plan_mass_tol=options.plan_mass_tol,
    mass_zero_tol=options.mass_zero_tol,
    top_n_heatmap=options.top_n_heatmap,
)

chosen_solution = tr.solve_unbalanced_transport(
    total_cost,
    healthy_mass,
    disease_mass,
    chosen_options.lambda_expand,
    chosen_options.lambda_reduce,
    chosen_options.lambda_residual,
)
chosen_components = tr.objective_components(
    chosen_solution,
    total_cost,
    chosen_options.lambda_expand,
    chosen_options.lambda_reduce,
    chosen_options.lambda_residual,
)
chosen_source_summary = tr.summarize_sources(
    chosen_solution,
    total_cost,
    healthy_df,
    disease_df,
    healthy_mass,
    chosen_options,
)
chosen_target_summary = tr.summarize_targets(
    chosen_solution,
    total_cost,
    healthy_df,
    disease_df,
    disease_mass,
    chosen_options,
)
chosen_plan_df = tr.build_plan_df(
    chosen_solution,
    cost_long,
    healthy_df,
    disease_df,
    healthy_mass,
    disease_mass,
    total_cost,
    chosen_options,
)

chosen_plan_df.to_csv(CHOSEN_OUT_DIR / 'transport_plan.csv', index=False)
chosen_source_summary.to_csv(CHOSEN_OUT_DIR / 'healthy_source_transport_summary.csv', index=False)
chosen_target_summary.to_csv(CHOSEN_OUT_DIR / 'disease_target_transport_summary.csv', index=False)
chosen_source_summary.to_csv(CHOSEN_OUT_DIR / 'transport_event_ranking.csv', index=False)
cost_long.to_csv(CHOSEN_OUT_DIR / 'cost_matrix.csv', index=False)

tr.write_json(
    CHOSEN_OUT_DIR / 'transport_parameters.json',
    {
        'run_name': f'{RUN_NAME}_source_regularized_lambda_{lambda_tag}',
        'parent_run': RUN_NAME,
        'organ': CONFIG['organ'],
        'healthy_clustering_method': CONFIG['healthy_clustering_method'],
        'disease_clustering_method': CONFIG['disease_clustering_method'],
        'healthy_cluster_csv': str(CONFIG['healthy_cluster_csv']),
        'disease_cluster_csv': str(CONFIG['disease_cluster_csv']),
        'out_dir': str(CHOSEN_OUT_DIR),
        'options': asdict(chosen_options),
        'cost_metadata': cost_metadata,
        'solver': {
            'objective': chosen_solution.objective,
            'status': chosen_solution.solver_status,
            'message': chosen_solution.solver_message,
        },
        'objective_components': chosen_components,
        'n_healthy_sources': int(len(healthy_df)),
        'n_disease_targets': int(len(disease_df)),
        'n_composition_features': int(len(comp_cols)),
        'composition_features': [c.replace('Comp_', '', 1) for c in comp_cols],
        'harmonization': {
            'defined_in_notebook': True,
            'canonical_map_size': int(len(canonical_cell_map)),
            'canonical_map': canonical_cell_map,
        },
    },
)

source_cols = [
    'healthy_cluster_id', 'healthy_mass', 'transported_out_mass', 'net_shift_mass',
    'log2_transport_shift', 'mean_transport_cost', 'cost_weighted_mass',
    'n_disease_branches', 'split_entropy', 'event_labels', 'top_disease_targets',
]
target_cols = [
    'disease_cluster_id', 'disease_mass', 'residual_mass', 'residual_fraction',
    'dominant_healthy_source', 'mean_incoming_cost', 'effective_n_sources', 'disease_signature',
]

print('Chosen diagnostic outputs:', CHOSEN_OUT_DIR)
display(chosen_source_summary[source_cols].head(10))
display(chosen_target_summary[target_cols].head(10))

## 12. Penalty Sweep Summary

This section summarizes the behavior of the transport model across the source-deviation penalty sweep. It uses only sweep-level outputs and is meant to answer how the OT formulation behaves as `lambda_expand = lambda_reduce` changes.

In [ ]:
import matplotlib.pyplot as plt


def ordered_unique(values, limit=None):
    seen = []
    for value in values:
        value = str(value)
        if value not in seen:
            seen.append(value)
        if limit is not None and len(seen) >= limit:
            break
    return seen


def values_close_to(requested_values, available_values):
    selected = []
    for requested in requested_values:
        matches = [value for value in available_values if np.isclose(value, requested)]
        if matches:
            selected.append(matches[0])
    return selected


def padded_limits(values, include_zero=False, floor=None, min_span=0.05):
    values = pd.Series(values, dtype='float64').replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        low, high = 0.0, min_span
    else:
        low = float(values.min())
        high = float(values.max())
    if include_zero:
        low = min(low, 0.0)
        high = max(high, 0.0)
    if floor is not None:
        low = min(low, floor)
    span = max(high - low, min_span)
    pad = span * 0.08
    return low - pad, high + pad


source_path_plot = source_path_df.copy()
source_path_plot['healthy_cluster_id'] = source_path_plot['healthy_cluster_id'].astype(str)
source_path_plot['positive_net_shift_mass'] = source_path_plot['net_shift_mass'].clip(lower=0)
source_path_plot['abs_net_shift_mass'] = source_path_plot['net_shift_mass'].abs()

target_path_plot = target_path_df.copy()
target_path_plot['disease_cluster_id'] = target_path_plot['disease_cluster_id'].astype(str)

sweep_source_scores = (
    source_path_plot
    .groupby('healthy_cluster_id', as_index=False)
    .agg(
        max_positive_net_shift=('positive_net_shift_mass', 'max'),
        max_abs_net_shift=('abs_net_shift_mass', 'max'),
        max_cost_weighted_mass=('cost_weighted_mass', 'max'),
        max_split_entropy=('split_entropy', 'max'),
        max_transported_out_mass=('transported_out_mass', 'max'),
        observed_lambdas=('lambda_shift', 'nunique'),
    )
)
sweep_source_scores['_source_sort'] = pd.to_numeric(sweep_source_scores['healthy_cluster_id'], errors='coerce')
sweep_source_scores['_source_sort'] = sweep_source_scores['_source_sort'].fillna(np.inf)


def top_sweep_source_ids(score_frame, metrics, n=3, limit=8):
    ids = []
    for metric in metrics:
        ids.extend(
            score_frame
            .sort_values([metric, '_source_sort', 'healthy_cluster_id'], ascending=[False, True, True])
            ['healthy_cluster_id']
            .head(n)
            .tolist()
        )
    return ordered_unique(ids, limit=limit)


sweep_tracked_sources = top_sweep_source_ids(
    sweep_source_scores,
    [
        'max_positive_net_shift',
        'max_abs_net_shift',
        'max_cost_weighted_mass',
        'max_split_entropy',
    ],
    n=3,
    limit=8,
)

lambda_values = lambda_path_df['lambda_shift'].tolist()
lambda_positions = np.arange(len(lambda_values))
lambda_labels = [f'{v:g}' for v in lambda_values]
REPRESENTATIVE_SWEEP_LAMBDAS = [0.0, 0.01, 0.05, 0.10, 0.20, 0.35]
sweep_landscape_lambdas = values_close_to(REPRESENTATIVE_SWEEP_LAMBDAS, lambda_values)


In [ ]:
sweep_source_scores

In [ ]:
sweep_tracked_sources

### 12.1 Sweep Mass Budget

Track whether the model uses expansion, reduction, or residual disease mass as the source-deviation penalty changes.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.5), constrained_layout=True)
fig.suptitle(
    f'{RUN_NAME}: Penalty Sweep Mass Budget\n'
    f'lambda_residual={PATH_RESIDUAL:g}; lambda_expand=lambda_reduce varied',
    fontsize=14,
)

for col, label, color in [
    ('expansion_mass', 'expansion', '#5477C4'),
    ('reduction_mass', 'reduction', '#CC6F47'),
    ('residual_mass', 'residual', '#71B436'),
]:
    ax.plot(lambda_positions, lambda_path_df[col], marker='o', linewidth=2, label=label, color=color)

ax.set_xlabel('lambda_expand = lambda_reduce')
ax.set_ylabel('total slack mass')
ax.set_xticks(lambda_positions)
ax.set_xticklabels(lambda_labels, rotation=45, ha='right')
ax.grid(axis='y', color='#E6E8F0', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, ncol=3)
plt.show()


### 12.2 Sweep Source Stability

Sources in this plot are selected from the full penalty sweep, not from the chosen diagnostic run. The notebook takes the union of the top three healthy sources by maximum positive net shift, maximum absolute net shift, maximum cost-weighted mass, and maximum split entropy across all `lambda_shift` values, then keeps the first eight unique sources.

In [ ]:
fig, ax = plt.subplots(figsize=(10.8, 5.2), constrained_layout=True)
fig.suptitle(f'{RUN_NAME}: sweep-selected source stability', fontsize=14)

line_colors = ['#5477C4', '#CC6F47', '#71B436', '#BD569B', '#B8A037', '#7A828F', '#2E4780', '#804126']
for idx, source_id in enumerate(sweep_tracked_sources):
    subset = (
        source_path_plot[source_path_plot['healthy_cluster_id'] == source_id]
        .sort_values('lambda_shift')
    )
    if subset.empty:
        continue
    ax.plot(
        lambda_positions,
        subset['net_shift_mass'],
        marker='o',
        linewidth=1.8,
        alpha=0.85,
        color=line_colors[idx % len(line_colors)],
        label=source_id,
    )
ax.axhline(0, color='#1F2430', linewidth=0.8)
ax.set_title('Tracked Source Net-Shift Trajectories')
ax.set_xlabel('lambda_expand = lambda_reduce')
ax.set_ylabel('net shift mass')
ax.set_xticks(lambda_positions)
ax.set_xticklabels(lambda_labels, rotation=45, ha='right')
ax.grid(axis='y', color='#E6E8F0', linewidth=0.8)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(title='healthy source', frameon=False, fontsize=8, title_fontsize=9, ncol=2)
plt.show()


### 12.3 Sweep Source Event Landscapes

Each panel is the source-event landscape for one representative `lambda_shift`. Points are healthy sources; x-position shows net expansion/reduction, y-position shows mean transport cost, size shows transported source mass, and color shows fragmentation entropy.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True, constrained_layout=True)
axes = axes.ravel()
fig.suptitle(
    f'{RUN_NAME}: source event landscapes across penalty sweep\n'
    f'lambda_residual={PATH_RESIDUAL:g}',
    fontsize=14,
)

source_landscape = source_path_plot[source_path_plot['lambda_shift'].isin(sweep_landscape_lambdas)].copy()
xlim = padded_limits(source_landscape['net_shift_mass'], include_zero=True, min_span=0.05)
ylim = padded_limits(source_landscape['mean_transport_cost'], floor=0.0, min_span=0.05)
max_source_mass = max(float(source_landscape['transported_out_mass'].max()), 1e-12)
source_color_max = max(float(source_landscape['split_entropy'].fillna(0.0).max()), 1e-12)
source_scatter = None

for idx, lam in enumerate(sweep_landscape_lambdas):
    ax = axes[idx]
    subset = source_landscape[np.isclose(source_landscape['lambda_shift'], lam)].copy()
    subset['plot_cost'] = subset['mean_transport_cost'].fillna(0.0)
    sizes = 35 + 520 * subset['transported_out_mass'] / max_source_mass
    source_scatter = ax.scatter(
        subset['net_shift_mass'],
        subset['plot_cost'],
        s=sizes,
        c=subset['split_entropy'].fillna(0.0),
        cmap='viridis',
        vmin=0.0,
        vmax=source_color_max,
        alpha=0.76,
        edgecolor='white',
        linewidth=0.6,
    )
    ax.axvline(0, color='#1F2430', linewidth=0.8)
    ax.set_title(f'lambda_shift={lam:g}')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(color='#E6E8F0', linewidth=0.7)
    ax.spines[['top', 'right']].set_visible(False)

    label_ids = ordered_unique(
        subset.sort_values(['net_shift_mass', 'healthy_cluster_id'], ascending=[False, True])['healthy_cluster_id'].head(3).tolist()
        + subset.sort_values(['cost_weighted_mass', 'healthy_cluster_id'], ascending=[False, True])['healthy_cluster_id'].head(3).tolist(),
        limit=5,
    )
    for _, row in subset[subset['healthy_cluster_id'].isin(label_ids)].iterrows():
        ax.annotate(
            str(row['healthy_cluster_id']),
            (row['net_shift_mass'], row['plot_cost']),
            fontsize=8,
            xytext=(4, 4),
            textcoords='offset points',
        )

for ax in axes[len(sweep_landscape_lambdas):]:
    ax.set_axis_off()
for ax in axes[::3]:
    ax.set_ylabel('mean transport cost')
for ax in axes[-3:]:
    ax.set_xlabel('net shift mass')
if source_scatter is not None:
    fig.colorbar(source_scatter, ax=axes[:len(sweep_landscape_lambdas)].tolist(), fraction=0.025, pad=0.02, label='split entropy')
plt.show()


### 12.4 Sweep Disease Target Explanation Landscapes

Each panel is the disease-target explanation landscape for one representative `lambda_shift`. Points are disease targets; x-position shows mixed-source structure, y-position shows residual disease fraction, size shows target mass, and color shows mean incoming transport cost.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True, constrained_layout=True)
axes = axes.ravel()
fig.suptitle(
    f'{RUN_NAME}: disease target explanation across penalty sweep\n'
    f'lambda_residual={PATH_RESIDUAL:g}',
    fontsize=14,
)

target_landscape = target_path_plot[target_path_plot['lambda_shift'].isin(sweep_landscape_lambdas)].copy()
target_landscape['plot_cost'] = target_landscape['mean_incoming_cost'].fillna(0.0)
xlim = padded_limits(target_landscape['effective_n_sources'], floor=0.0, min_span=0.25)
ymax = float(target_landscape['residual_fraction'].fillna(0.0).max())
ylim = (-0.01, max(ymax * 1.15, 0.05))
max_target_mass = max(float(target_landscape['disease_mass'].max()), 1e-12)
target_color_max = max(float(target_landscape['plot_cost'].max()), 1e-12)
target_scatter = None

for idx, lam in enumerate(sweep_landscape_lambdas):
    ax = axes[idx]
    subset = target_landscape[np.isclose(target_landscape['lambda_shift'], lam)].copy()
    sizes = 35 + 520 * subset['disease_mass'] / max_target_mass
    target_scatter = ax.scatter(
        subset['effective_n_sources'],
        subset['residual_fraction'],
        s=sizes,
        c=subset['plot_cost'],
        cmap='plasma',
        vmin=0.0,
        vmax=target_color_max,
        alpha=0.76,
        edgecolor='white',
        linewidth=0.6,
    )
    ax.set_title(f'lambda_shift={lam:g}')
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(color='#E6E8F0', linewidth=0.7)
    ax.spines[['top', 'right']].set_visible(False)
    if ymax <= 1e-8:
        ax.text(0.04, 0.92, 'no residual mass', transform=ax.transAxes, fontsize=8, va='top')

    label_ids = ordered_unique(
        subset.sort_values(['residual_fraction', 'disease_cluster_id'], ascending=[False, True])['disease_cluster_id'].head(3).tolist()
        + subset.sort_values(['effective_n_sources', 'disease_cluster_id'], ascending=[False, True])['disease_cluster_id'].head(3).tolist(),
        limit=5,
    )
    for _, row in subset[subset['disease_cluster_id'].isin(label_ids)].iterrows():
        ax.annotate(
            str(row['disease_cluster_id']),
            (row['effective_n_sources'], row['residual_fraction']),
            fontsize=8,
            xytext=(4, 4),
            textcoords='offset points',
        )

for ax in axes[len(sweep_landscape_lambdas):]:
    ax.set_axis_off()
for ax in axes[::3]:
    ax.set_ylabel('residual fraction')
for ax in axes[-3:]:
    ax.set_xlabel('effective number of sources')
if target_scatter is not None:
    fig.colorbar(target_scatter, ax=axes[:len(sweep_landscape_lambdas)].tolist(), fraction=0.025, pad=0.02, label='mean incoming cost')
plt.show()


## 13. Chosen Diagnostic Run Inspection

This section inspects one concrete transport plan after the sweep-level behavior has been reviewed. These plots are conditional on the chosen diagnostic penalties and should be used for follow-up event selection, not for judging the global penalty pattern.

In [ ]:
chosen_run_label = (
    f'lambda_expand=lambda_reduce={chosen_options.lambda_expand:g}; '
    f'lambda_residual={chosen_options.lambda_residual:g}'
)

selected_source_id = str(
    chosen_source_summary
    .sort_values(['net_shift_mass', 'cost_weighted_mass', 'transported_out_mass', 'healthy_cluster_id'], ascending=[False, False, False, True])
    .iloc[0]['healthy_cluster_id']
)
selected_source_row = chosen_source_summary[chosen_source_summary['healthy_cluster_id'].astype(str) == selected_source_id].iloc[0]
selected_source_links = (
    chosen_plan_df[chosen_plan_df['healthy_cluster_id'].astype(str) == selected_source_id]
    .sort_values('transport_mass', ascending=False)
)

summary_takeaways = pd.DataFrame({
    'quantity': [
        'chosen_expansion_mass',
        'chosen_reduction_mass',
        'chosen_residual_mass',
        'selected_source',
        'selected_net_shift_mass',
        'selected_mean_transport_cost',
        'selected_disease_branches',
    ],
    'value': [
        chosen_components['expansion_mass'],
        chosen_components['reduction_mass'],
        chosen_components['residual_mass'],
        selected_source_id,
        selected_source_row['net_shift_mass'],
        selected_source_row['mean_transport_cost'],
        selected_source_row['n_disease_branches'],
    ],
})
display(summary_takeaways)


### 13.1 Chosen-Run Slack Mass

Show how much expansion, reduction, and residual disease mass is used by the selected diagnostic run.

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.3), constrained_layout=True)
fig.suptitle(f'{RUN_NAME}: chosen-run slack mass\n{chosen_run_label}', fontsize=14)

slack_names = ['expansion', 'reduction', 'residual']
slack_values = np.array([chosen_components[f'{name}_mass'] for name in slack_names], dtype=float)
colors = ['#5477C4', '#CC6F47', '#71B436']
ax.bar(slack_names, slack_values, color=colors, edgecolor='#FFFFFF', linewidth=0.8)
ymax = max(float(slack_values.max()) * 1.25, 0.01)
ax.set_ylim(0, ymax)
ax.set_ylabel('mass')
ax.set_title('Slack Mass Used By Chosen Run')
ax.spines[['top', 'right']].set_visible(False)
for idx, value in enumerate(slack_values):
    ax.text(idx, max(value, ymax * 0.04), f'{value:.3f}', ha='center', va='bottom', fontsize=9)
plt.show()


### 13.2 Chosen-Run Source Event Landscape

Inspect source-level expansion, cost, and fragmentation for the selected diagnostic run.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2), constrained_layout=True)
fig.suptitle(f'{RUN_NAME}: chosen-run source event landscape\n{chosen_run_label}', fontsize=14)

source_plot = chosen_source_summary.copy()
max_source_mass = max(float(source_plot['transported_out_mass'].max()), 1e-12)
source_sizes = 60 + 900 * source_plot['transported_out_mass'] / max_source_mass
source_scatter = ax.scatter(
    source_plot['net_shift_mass'],
    source_plot['mean_transport_cost'],
    s=source_sizes,
    c=source_plot['split_entropy'],
    cmap='viridis',
    alpha=0.78,
    edgecolor='white',
    linewidth=0.6,
)
ax.axvline(0, color='#1F2430', linewidth=0.8)
ax.set_title('Source Event Landscape')
ax.set_xlabel('net shift mass')
ax.set_ylabel('mean transport cost')
ax.spines[['top', 'right']].set_visible(False)
fig.colorbar(source_scatter, ax=ax, fraction=0.046, pad=0.04, label='split entropy')
label_sources = (
    source_plot
    .sort_values(['net_shift_mass', 'cost_weighted_mass', 'transported_out_mass'], ascending=[False, False, False])
    .head(7)
)
for _, row in label_sources.iterrows():
    ax.annotate(str(row['healthy_cluster_id']), (row['net_shift_mass'], row['mean_transport_cost']), fontsize=8, xytext=(4, 4), textcoords='offset points')
plt.show()


### 13.3 Chosen-Run Transport Structure

Inspect the selected source's disease branches and the largest links in the chosen global transport plan.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
fig.suptitle(f'{RUN_NAME}: chosen-run transport structure for source {selected_source_id}\n{chosen_run_label}', fontsize=14)

ax = axes[0]
branch_plot = selected_source_links.head(12).iloc[::-1]
if branch_plot.empty:
    ax.text(0.5, 0.5, 'No outgoing links for selected source', ha='center', va='center')
    ax.set_axis_off()
else:
    labels = branch_plot['disease_cluster_id'].astype(str)
    ax.barh(labels, branch_plot['transport_mass'], color='#5477C4')
    ax.set_title('Selected Source Disease Branches')
    ax.set_xlabel('transport mass')
    ax.set_ylabel('disease target')
    ax.spines[['top', 'right']].set_visible(False)
    for _, row in branch_plot.iterrows():
        ax.text(row['transport_mass'], str(row['disease_cluster_id']), f" cost={row['total_cost']:.2f}", va='center', ha='left', fontsize=8)

ax = axes[1]
heatmap_links = chosen_plan_df.sort_values('transport_mass', ascending=False).head(30)
if heatmap_links.empty:
    ax.text(0.5, 0.5, 'No nonzero transport links', ha='center', va='center')
    ax.set_axis_off()
else:
    h_order = heatmap_links.groupby('healthy_cluster_id')['transport_mass'].sum().sort_values(ascending=False).index.astype(str).tolist()
    d_order = heatmap_links.groupby('disease_cluster_id')['transport_mass'].sum().sort_values(ascending=False).index.astype(str).tolist()
    heatmap = (
        heatmap_links
        .pivot_table(index='healthy_cluster_id', columns='disease_cluster_id', values='transport_mass', aggfunc='sum', fill_value=0.0)
        .reindex(index=h_order, columns=d_order, fill_value=0.0)
    )
    values = np.ma.masked_where(heatmap.to_numpy() <= 0, heatmap.to_numpy())
    cmap = plt.get_cmap('YlOrRd').copy()
    cmap.set_bad('white')
    im = ax.imshow(values, aspect='auto', cmap=cmap)
    ax.set_title('Largest Global Transport Links')
    ax.set_xlabel('disease target')
    ax.set_ylabel('healthy source')
    ax.set_xticks(range(len(d_order)))
    ax.set_xticklabels(d_order, rotation=90, fontsize=8)
    ax.set_yticks(range(len(h_order)))
    ax.set_yticklabels(h_order, fontsize=8)
    ax.set_xticks(np.arange(-.5, len(d_order), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(h_order), 1), minor=True)
    ax.grid(which='minor', color='#dddddd', linestyle='-', linewidth=0.5)
    ax.tick_params(which='minor', bottom=False, left=False)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='transport mass')
plt.show()


### 13.4 Chosen-Run Disease Target Explanation

Inspect whether disease targets are residual or mixed-source under the selected diagnostic run.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2), constrained_layout=True)
fig.suptitle(f'{RUN_NAME}: chosen-run disease target explanation\n{chosen_run_label}', fontsize=14)

target_plot = chosen_target_summary.copy()
max_target_mass = max(float(target_plot['disease_mass'].max()), 1e-12)
target_sizes = 60 + 900 * target_plot['disease_mass'] / max_target_mass
target_color = target_plot['mean_incoming_cost'].fillna(0.0)
target_scatter = ax.scatter(
    target_plot['effective_n_sources'],
    target_plot['residual_fraction'],
    s=target_sizes,
    c=target_color,
    cmap='plasma',
    alpha=0.78,
    edgecolor='white',
    linewidth=0.6,
)
resid_max = float(target_plot['residual_fraction'].fillna(0).max())
if resid_max <= 1e-8:
    ax.set_ylim(-0.01, 0.05)
    ax.text(0.02, 0.93, 'No substantial residual disease mass', transform=ax.transAxes, fontsize=9, va='top')
else:
    ax.set_ylim(0, resid_max * 1.2)
ax.set_title('Disease Target Explanation')
ax.set_xlabel('effective number of sources')
ax.set_ylabel('residual fraction')
ax.spines[['top', 'right']].set_visible(False)
fig.colorbar(target_scatter, ax=ax, fraction=0.046, pad=0.04, label='mean incoming cost')
label_targets = pd.concat([
    target_plot.sort_values('residual_fraction', ascending=False).head(3),
    target_plot.sort_values('effective_n_sources', ascending=False).head(4),
]).drop_duplicates('disease_cluster_id')
for _, row in label_targets.iterrows():
    ax.annotate(str(row['disease_cluster_id']), (row['effective_n_sources'], row['residual_fraction']), fontsize=8, xytext=(4, 4), textcoords='offset points')
plt.show()


## 14. Selected Source Composition Remodeling Across Sweep

This section selects the most stable positively shifted healthy source from the non-degenerate penalty sweep, then measures its transport-weighted disease composition across the full sweep from `lambda_shift = 0` to `0.35`. The collapsed high-penalty endpoint is shown for context, but source selection is not based on collapsed shifts.


In [ ]:
SWEEP_COMPOSITION_OUT_DIR = OUT_DIR / 'selected_source_composition_remodeling'
SWEEP_COMPOSITION_OUT_DIR.mkdir(parents=True, exist_ok=True)

selection_tol = max(float(options.mass_zero_tol), 1e-10)
composition_sweep_lambdas = sorted(lambda_path_df['lambda_shift'].astype(float).unique())

source_selection_df = source_path_df.copy()
source_selection_df['healthy_cluster_id'] = source_selection_df['healthy_cluster_id'].astype(str)
source_selection_df['positive_net_shift_mass'] = source_selection_df['net_shift_mass'].clip(lower=0.0)
source_selection_df['_source_sort'] = pd.to_numeric(source_selection_df['healthy_cluster_id'], errors='coerce').fillna(np.inf)

lambda_signal = source_selection_df.groupby('lambda_shift', as_index=False).agg(
    max_positive_net_shift=('positive_net_shift_mass', 'max')
)
source_selection_lambdas = lambda_signal.loc[
    lambda_signal['max_positive_net_shift'] > selection_tol,
    'lambda_shift',
].sort_values().tolist()
if not source_selection_lambdas:
    raise RuntimeError('No non-degenerate positive source-shift lambdas were found for source selection.')

selection_valid = source_selection_df[source_selection_df['lambda_shift'].isin(source_selection_lambdas)].copy()
selection_valid['positive_rank'] = np.nan
positive_order = (
    selection_valid[selection_valid['positive_net_shift_mass'] > selection_tol]
    .sort_values(['lambda_shift', 'positive_net_shift_mass', '_source_sort', 'healthy_cluster_id'], ascending=[True, False, True, True])
)
selection_valid.loc[positive_order.index, 'positive_rank'] = positive_order.groupby('lambda_shift').cumcount() + 1
selection_valid['is_rank1_positive'] = selection_valid['positive_rank'].eq(1)
selection_valid['is_top3_positive'] = selection_valid['positive_rank'].le(3)

stable_source_scores = (
    selection_valid
    .groupby('healthy_cluster_id', as_index=False)
    .agg(
        n_selection_lambdas=('lambda_shift', 'nunique'),
        n_rank1_positive=('is_rank1_positive', 'sum'),
        n_top3_positive=('is_top3_positive', 'sum'),
        mean_positive_net_shift=('positive_net_shift_mass', 'mean'),
        median_positive_net_shift=('positive_net_shift_mass', 'median'),
        max_positive_net_shift=('positive_net_shift_mass', 'max'),
        max_cost_weighted_mass=('cost_weighted_mass', 'max'),
        max_split_entropy=('split_entropy', 'max'),
        mean_transport_cost=('mean_transport_cost', 'mean'),
    )
)
stable_source_scores['rank1_fraction'] = stable_source_scores['n_rank1_positive'] / len(source_selection_lambdas)
stable_source_scores['top3_fraction'] = stable_source_scores['n_top3_positive'] / len(source_selection_lambdas)
stable_source_scores['_source_sort'] = pd.to_numeric(stable_source_scores['healthy_cluster_id'], errors='coerce').fillna(np.inf)
stable_source_scores = stable_source_scores.sort_values(
    ['n_rank1_positive', 'n_top3_positive', 'mean_positive_net_shift', 'max_positive_net_shift', '_source_sort'],
    ascending=[False, False, False, False, True],
).drop(columns='_source_sort')

sweep_selected_source_id = str(stable_source_scores.iloc[0]['healthy_cluster_id'])
selection_summary = stable_source_scores.head(10).copy()

print('source-selection lambdas:', [f'{x:g}' for x in source_selection_lambdas])
print('composition-plot lambdas:', [f'{x:g}' for x in composition_sweep_lambdas])
print('sweep-selected source:', sweep_selected_source_id)
stable_source_scores


In [ ]:
healthy_ids = healthy_df['cluster_id'].astype(str).tolist()
disease_ids = disease_df['cluster_id'].astype(str).tolist()
healthy_index = {cluster_id: idx for idx, cluster_id in enumerate(healthy_ids)}
disease_index = {cluster_id: idx for idx, cluster_id in enumerate(disease_ids)}
selected_source_idx = healthy_index[sweep_selected_source_id]
selected_healthy_profile = pd.Series(healthy_comp[selected_source_idx], index=comp_cols, dtype='float64')


def source_row_at_lambda(frame, source_id, lam):
    rows = frame[np.isclose(frame['lambda_shift'], lam) & frame['healthy_cluster_id'].astype(str).eq(str(source_id))]
    if rows.empty:
        raise RuntimeError(f'Missing source {source_id} at lambda_shift={lam:g}.')
    return rows.iloc[0]


def selected_source_links_at_lambda(lam):
    solution_lam = tr.solve_unbalanced_transport(total_cost, healthy_mass, disease_mass, lam, lam, PATH_RESIDUAL)
    plan_lam = tr.build_plan_df(solution_lam, cost_long, healthy_df, disease_df, healthy_mass, disease_mass, total_cost, options)
    links = plan_lam[plan_lam['healthy_cluster_id'].astype(str).eq(sweep_selected_source_id)].copy()
    return links[links['transport_mass'] > float(options.plan_mass_tol)].copy()


def transport_weighted_disease_profile(links):
    target_idx = [disease_index[str(cluster_id)] for cluster_id in links['disease_cluster_id'].astype(str)]
    weights = links['transport_mass'].to_numpy(dtype=float)
    return pd.Series(np.average(disease_comp[target_idx, :], axis=0, weights=weights), index=comp_cols, dtype='float64')


def sign_consistency(values, tol=1e-8):
    vals = pd.Series(values, dtype='float64').dropna()
    if vals.empty:
        return np.nan
    mean_delta = float(vals.mean())
    if mean_delta > tol:
        return float((vals > tol).mean())
    if mean_delta < -tol:
        return float((vals < -tol).mean())
    return float((vals.abs() <= tol).mean())


selected_links = []
composition_rows = []
for lam in composition_sweep_lambdas:
    source_event = source_row_at_lambda(source_selection_df, sweep_selected_source_id, lam)
    rank_event = source_row_at_lambda(selection_valid, sweep_selected_source_id, lam) if lam in source_selection_lambdas else None
    source_links = selected_source_links_at_lambda(lam)
    if source_links.empty:
        continue

    is_selection_lambda = lam in source_selection_lambdas
    source_links.insert(0, 'lambda_shift', lam)
    source_links.insert(1, 'selected_source_id', sweep_selected_source_id)
    source_links['is_source_selection_lambda'] = is_selection_lambda
    source_links['selected_source_net_shift_mass'] = float(source_event['net_shift_mass'])
    source_links['selected_source_positive_rank'] = rank_event['positive_rank'] if rank_event is not None else np.nan
    source_links['selected_source_mean_transport_cost'] = float(source_event['mean_transport_cost'])
    selected_links.append(source_links)

    disease_profile = transport_weighted_disease_profile(source_links)
    for comp_col in comp_cols:
        healthy_fraction = float(selected_healthy_profile[comp_col])
        disease_fraction = float(disease_profile[comp_col])
        composition_rows.append({
            'lambda_shift': lam,
            'selected_source_id': sweep_selected_source_id,
            'is_source_selection_lambda': is_selection_lambda,
            'cell_type': comp_col.replace('Comp_', '', 1),
            'healthy_fraction': healthy_fraction,
            'transport_weighted_disease_fraction': disease_fraction,
            'delta': disease_fraction - healthy_fraction,
            'abs_delta': abs(disease_fraction - healthy_fraction),
            'transported_out_mass': float(source_links['transport_mass'].sum()),
            'net_shift_mass': float(source_event['net_shift_mass']),
            'positive_rank': rank_event['positive_rank'] if rank_event is not None else np.nan,
            'mean_transport_cost': float(source_event['mean_transport_cost']),
            'n_disease_targets': int(source_links['disease_cluster_id'].nunique()),
        })

if not selected_links or not composition_rows:
    raise RuntimeError(f'No sweep composition outputs were produced for selected source {sweep_selected_source_id}.')

selected_source_sweep_links = pd.concat(selected_links, ignore_index=True)
composition_change_by_lambda = pd.DataFrame(composition_rows)
composition_change_stability_summary = (
    composition_change_by_lambda
    .groupby('cell_type', as_index=False)
    .agg(
        mean_delta=('delta', 'mean'),
        median_delta=('delta', 'median'),
        min_delta=('delta', 'min'),
        max_delta=('delta', 'max'),
        mean_abs_delta=('abs_delta', 'mean'),
        max_abs_delta=('abs_delta', 'max'),
        mean_healthy_fraction=('healthy_fraction', 'mean'),
        mean_transport_weighted_disease_fraction=('transport_weighted_disease_fraction', 'mean'),
        n_lambdas=('lambda_shift', 'nunique'),
    )
)
composition_change_stability_summary['sign_consistency_fraction'] = (
    composition_change_by_lambda.groupby('cell_type')['delta']
    .apply(sign_consistency)
    .reindex(composition_change_stability_summary['cell_type'])
    .to_numpy()
)
composition_change_stability_summary['stable_abs_delta_score'] = (
    composition_change_stability_summary['mean_abs_delta']
    * composition_change_stability_summary['sign_consistency_fraction'].fillna(0.0)
)
composition_change_stability_summary = composition_change_stability_summary.sort_values(
    ['stable_abs_delta_score', 'mean_abs_delta', 'cell_type'],
    ascending=[False, False, True],
)

branch_recurrence_summary = (
    selected_source_sweep_links
    .groupby('disease_cluster_id', as_index=False)
    .agg(
        n_lambdas_linked=('lambda_shift', 'nunique'),
        n_selection_lambdas_linked=('is_source_selection_lambda', 'sum'),
        mean_transport_mass=('transport_mass', 'mean'),
        max_transport_mass=('transport_mass', 'max'),
        mean_total_cost=('total_cost', 'mean'),
        mean_fraction_of_source=('transport_fraction_of_healthy_source', 'mean'),
        mean_fraction_of_target=('transport_fraction_of_disease_target', 'mean'),
        disease_mass=('disease_mass', 'first'),
        disease_signature=('disease_signature', 'first'),
        disease_top_enriched=('disease_top_enriched', 'first'),
    )
    .sort_values(['n_lambdas_linked', 'mean_transport_mass', 'max_transport_mass'], ascending=[False, False, False])
)

selected_source_sweep_links.to_csv(SWEEP_COMPOSITION_OUT_DIR / 'selected_source_sweep_links.csv', index=False)
composition_change_by_lambda.to_csv(SWEEP_COMPOSITION_OUT_DIR / 'composition_change_by_lambda.csv', index=False)
composition_change_stability_summary.to_csv(SWEEP_COMPOSITION_OUT_DIR / 'composition_change_stability_summary.csv', index=False)

print('wrote selected-source composition outputs to:', SWEEP_COMPOSITION_OUT_DIR)
display(composition_change_stability_summary.head(15))
display(branch_recurrence_summary.head(12))


### 14.1 Composition Change Across The Sweep

Rows are all cell types with nonzero composition change across the sweep. Columns span the full plotted sweep from `lambda_shift = 0` to `0.35`. Positive values mean the transport-weighted disease branches have a higher fraction than the healthy source; negative values mean they have a lower fraction.


In [ ]:
NONZERO_DELTA_TOL = 1e-12
nonzero_composition_summary = composition_change_stability_summary[
    composition_change_stability_summary['max_abs_delta'] > NONZERO_DELTA_TOL
].copy()
if nonzero_composition_summary.empty:
    raise RuntimeError('No cell types have nonzero composition delta across the sweep.')
TOP_COMPOSITION_CELL_TYPES = len(nonzero_composition_summary)
top_composition_cell_types = nonzero_composition_summary['cell_type'].tolist()
heatmap_data = (
    composition_change_by_lambda[composition_change_by_lambda['cell_type'].isin(top_composition_cell_types)]
    .pivot_table(index='cell_type', columns='lambda_shift', values='delta', aggfunc='mean')
    .reindex(index=top_composition_cell_types, columns=composition_sweep_lambdas)
)
heatmap_values = np.ma.masked_invalid(heatmap_data.to_numpy(dtype=float))
max_abs_delta = max(float(np.nanmax(np.abs(heatmap_data.to_numpy(dtype=float)))), 1e-6)

fig, ax = plt.subplots(figsize=(11.5, max(5.4, 0.34 * len(top_composition_cell_types) + 1.8)), constrained_layout=True)
fig.suptitle(
    f'{RUN_NAME}: selected source {sweep_selected_source_id} composition remodeling across full sweep\n'
    f'lambda_shift range {composition_sweep_lambdas[0]:g} to {composition_sweep_lambdas[-1]:g}; lambda_residual={PATH_RESIDUAL:g}',
    fontsize=14,
)
cmap = plt.get_cmap('RdBu_r').copy()
cmap.set_bad('#F0F1F4')
im = ax.imshow(heatmap_values, aspect='auto', cmap=cmap, vmin=-max_abs_delta, vmax=max_abs_delta)
ax.set_title('Transport-weighted disease composition minus healthy source composition')
ax.set_xlabel('lambda_expand = lambda_reduce')
ax.set_ylabel('cell type')
ax.set_xticks(np.arange(len(composition_sweep_lambdas)))
ax.set_xticklabels([f'{x:g}' for x in composition_sweep_lambdas], rotation=45, ha='right')
ax.set_yticks(np.arange(len(top_composition_cell_types)))
ax.set_yticklabels(top_composition_cell_types)
ax.set_xlim(-0.5, len(composition_sweep_lambdas) - 0.5)
ax.set_xticks(np.arange(-0.5, len(composition_sweep_lambdas), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(top_composition_cell_types), 1), minor=True)
ax.grid(which='minor', color='#FFFFFF', linewidth=0.7)
ax.tick_params(which='minor', bottom=False, left=False)
fig.colorbar(im, ax=ax, fraction=0.032, pad=0.02, label='signed composition delta')
plt.show()


### 14.2 Stable Composition Remodeling Summary

This plot averages each nonzero cell-type delta equally across the full plotted sweep. The right panel shows how consistently the sign of each change agrees with the mean direction.


In [ ]:
summary_plot = nonzero_composition_summary.copy()
y = np.arange(len(summary_plot))
bar_colors = np.where(summary_plot['mean_delta'] >= 0, '#C15A4A', '#3E6DA9')

fig, (ax_delta, ax_consistency) = plt.subplots(
    1,
    2,
    figsize=(11.2, max(5.4, 0.38 * len(summary_plot) + 1.8)),
    gridspec_kw={'width_ratios': [3.2, 1.15]},
    sharey=True,
    constrained_layout=True,
)
fig.suptitle(f'{RUN_NAME}: selected source {sweep_selected_source_id} stable composition remodeling', fontsize=14)

ax_delta.barh(y, summary_plot['mean_delta'], color=bar_colors, edgecolor='white', linewidth=0.8)
ax_delta.axvline(0, color='#1F2430', linewidth=0.9)
ax_delta.set_yticks(y)
ax_delta.set_yticklabels(summary_plot['cell_type'], fontsize=9)
ax_delta.invert_yaxis()
ax_delta.set_title('Mean signed delta')
ax_delta.set_xlabel('composition delta')
ax_delta.set_ylabel('cell type')
ax_delta.grid(axis='x', color='#E6E8F0', linewidth=0.8)
ax_delta.spines[['top', 'right']].set_visible(False)

ax_consistency.scatter(
    summary_plot['sign_consistency_fraction'],
    y,
    s=70,
    c=bar_colors,
    edgecolor='white',
    linewidth=0.8,
)
ax_consistency.set_xlim(0, 1.05)
ax_consistency.set_xticks([0, 0.5, 1.0])
ax_consistency.set_xticklabels(['0%', '50%', '100%'])
ax_consistency.set_title('Sign consistency')
ax_consistency.set_xlabel('fraction')
ax_consistency.grid(axis='x', color='#E6E8F0', linewidth=0.8)
ax_consistency.tick_params(axis='y', left=False, labelleft=False)
ax_consistency.spines[['top', 'right', 'left']].set_visible(False)
plt.show()


### 14.3 Recurrent Disease Branches For The Selected Source

This table lists disease targets repeatedly linked to the selected healthy source across the full sweep. It is a branch-stability diagnostic, not a single-lambda branch definition.


In [ ]:
display(branch_recurrence_summary.head(12))
